# DBT Query Compiler

This notebook helps you compile and view dbt models locally to see what the compiled SQL looks like.

## 1. Import Required Libraries

In [43]:
import os
import subprocess
from pathlib import Path
from IPython.display import display, Markdown
import sqlparse

# Set the working directory to the transformation folder
os.chdir('/Users/tylerclifton/Projects/mbta-reliability-analytics/transformation')
print(f"Current directory: {os.getcwd()}")

Current directory: /Users/tylerclifton/Projects/mbta-reliability-analytics/transformation


## 2. Compile All Models

Run `dbt compile` to compile all models and macros in the project.

In [44]:
# Run dbt compile
# Using ! to run shell command in the notebook's kernel environment
!dbt compile

16:33:59  Running with dbt=1.7.8
16:34:00  Registered adapter: bigquery=1.7.4
16:34:00  Encountered an error:
Compilation Error
  unexpected ']', expected '}'
    line 91
      ]


## 3. View Compiled SQL

Helper function to read and display compiled SQL from the target directory.

In [45]:
def view_compiled_sql(model_path):
    """
    View the compiled SQL for a given model with formatted output.
    
    Args:
        model_path: Path to the model relative to models/ (e.g., 'mbta/bronze/mbta_bronze_alerts.sql')
    """
    compiled_path = f"target/compiled/mbta_reliability_analytics/models/{model_path}"
    
    if not os.path.exists(compiled_path):
        print(f"❌ Compiled file not found: {compiled_path}")
        return
    
    with open(compiled_path, 'r') as f:
        sql_content = f.read()
    
    # Format SQL with uppercase keywords, indentation, and proper spacing
    formatted_sql = sqlparse.format(
        sql_content,
        keyword_case='upper',
        identifier_case='lower',
        reindent=True,
        indent_width=2
    )
    
    print(f"📄 Compiled SQL for: {model_path}")
    print("=" * 80)
    display(Markdown(f"```sql\n{formatted_sql}\n```"))
    print("=" * 80)
    
# Test the function
print("Function defined. Use it like: view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')")

Function defined. Use it like: view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')


## 4. View MBTA Bronze Alerts Model

In [46]:
view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')

📄 Compiled SQL for: mbta/bronze/mbta_bronze_alerts.sql


```sql

SELECT active_period_start,
       active_period_end,
       alert_id,
       route,
       header,
       description,
       cause,
       effect,
       severity,
       duration_certainty,
       created_at,
       updated_at,
       ingestion_timestamp,
       ingestion_source
FROM `mbta-reliability-analytics.staging.mbta_alerts`
```

## 5. View MBTA Bronze Routes Model

In [47]:
view_compiled_sql('mbta/bronze/mbta_bronze_routes.sql')

📄 Compiled SQL for: mbta/bronze/mbta_bronze_routes.sql


```sql

SELECT route_id,
       long_name,
       description,
       route_type,
       color,
       direction_destinations,
       ingestion_timestamp,
       ingestion_source
FROM `mbta-reliability-analytics.staging.mbta_routes`
```

## 6. View MBTA Silver Model

In [48]:
view_compiled_sql('mbta/silver/mbta_silver.sql')

📄 Compiled SQL for: mbta/silver/mbta_silver.sql


```sql

SELECT -- Alert fields (from bronze_alerts)
 cast(regexp_extract(cast(a.active_period_start AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_start_date,
 cast(regexp_extract(cast(a.active_period_end AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_end_date,
 cast(a.alert_id AS STRING) AS alert_id,
 cast(a.route AS STRING) AS route_id,
 cast(a.header AS STRING) AS alert_header,
 cast(a.description AS STRING) AS alert_description,
 cast(a.cause AS STRING) AS alert_cause,
 cast(a.effect AS STRING) AS alert_effect,
 cast(a.severity AS int64) AS alert_severity,
 cast(a.duration_certainty AS STRING) AS alert_duration_certainty,
 cast(regexp_extract(cast(a.created_at AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_created_at,
 cast(regexp_extract(cast(a.updated_at AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_updated_at, -- Route fields (from bronze_routes)
 cast(r.long_name AS STRING) AS route_name,
 cast(r.description AS STRING) AS route_description,
 cast(r.route_type AS STRING) AS route_type,
 cast(r.color AS STRING) AS route_color,
 cast(r.direction_destinations AS STRING) AS route_destinations, -- Metadata
 cast(a.ingestion_source AS STRING) AS ingestion_source,
 cast(a.ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp
FROM `mbta-reliability-analytics`.`mbta_dev`.`mbta_bronze_alerts` AS a
LEFT JOIN `mbta-reliability-analytics`.`mbta_dev`.`mbta_bronze_routes` AS r ON a.route = r.route_id
```

## 7. View NWS Bronze Weather Model

In [49]:
view_compiled_sql('nws/bronze/nws_bronze_weather.sql')

📄 Compiled SQL for: nws/bronze/nws_bronze_weather.sql


```sql

SELECT observation_timestamp,
       station_id,
       temperature_fahrenheit,
       temperature_celsius,
       dewpoint_fahrenheit,
       dewpoint_celsius,
       wind_speed_mph,
       wind_direction_degrees,
       precipitation_last_hour_mm,
       relative_humidity_percent,
       barometric_pressure_pa,
       visibility_miles,
       cloud_base_meters,
       cloud_coverage,
       conditions,
       ingestion_timestamp,
       ingestion_source
FROM `mbta-reliability-analytics.staging.nws_weather`
```

## 8. View NWS Silver Weather Model

In [50]:
view_compiled_sql('nws/silver/nws_silver.sql')

📄 Compiled SQL for: nws/silver/nws_silver.sql


```sql

SELECT cast(observation_timestamp AS TIMESTAMP) AS observation_timestamp,
       cast(station_id AS STRING) AS station_id,
       cast(temperature_fahrenheit AS float64) AS temperature_fahrenheit,
       cast(temperature_celsius AS float64) AS temperature_celsius,
       cast(dewpoint_fahrenheit AS float64) AS dewpoint_fahrenheit,
       cast(dewpoint_celsius AS float64) AS dewpoint_celsius,
       cast(wind_speed_mph AS float64) AS wind_speed_mph,
       cast(wind_direction_degrees AS float64) AS wind_direction_degrees,
       cast(precipitation_last_hour_mm AS float64) AS precipitation_last_hour_mm,
       cast(relative_humidity_percent AS float64) AS relative_humidity_percent,
       cast(barometric_pressure_pa AS float64) AS barometric_pressure_pa,
       cast(visibility_miles AS float64) AS visibility_miles,
       cast(cloud_base_meters AS float64) AS cloud_base_meters,
       cast(cloud_coverage AS STRING) AS cloud_coverage,
       cast(conditions AS STRING) AS conditions,
       cast(ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp,
       cast(ingestion_source AS STRING) AS ingestion_source
FROM `mbta-reliability-analytics`.`mbta_dev`.`nws_bronze_weather`
```

## 9. List All Available Compiled Models

Use this to see all compiled SQL files in your target directory.

In [51]:
import glob

# Find all compiled SQL files
compiled_dir = "target/compiled/mbta_reliability_analytics/models/"
if os.path.exists(compiled_dir):
    sql_files = glob.glob(f"{compiled_dir}**/*.sql", recursive=True)
    
    print(f"Found {len(sql_files)} compiled SQL files:\n")
    for file_path in sorted(sql_files):
        # Get relative path from models directory
        rel_path = file_path.replace(compiled_dir, "")
        print(f"  • {rel_path}")
else:
    print(f"Compiled directory not found: {compiled_dir}")
    print("Run dbt compile first!")

Found 5 compiled SQL files:

  • mbta/bronze/mbta_bronze_alerts.sql
  • mbta/bronze/mbta_bronze_routes.sql
  • mbta/silver/mbta_silver.sql
  • nws/bronze/nws_bronze_weather.sql
  • nws/silver/nws_silver.sql


## 10. View Any Custom Model

Use the function below to view any compiled model by specifying its path.

In [52]:
# Example: View any model by specifying its path
# view_compiled_sql('path/to/your/model.sql')

# Uncomment and modify the line below to view a specific model:
# view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')